# Deep Agents Practice Notebook

Practice the **Deep Agents SDK** core capabilities from the [official overview](https://docs.langchain.com/oss/python/deepagents/overview#core-capabilities).

**Model:** `openrouter:openai/gpt-6-sol`  
**Auth:** `OPENROUTER_API_KEY` in `.env` (no Tavily / second search vendor)

| Part | Capability group |
|------|------------------|
| 0 | Setup |
| 1 | Harness overview + quickstart |
| 2 | **Execution environment** — tools, filesystem, permissions, interpreter, streaming |
| 3 | **Context management** — skills, memory, offloading, prompt caching |
| 4 | **Delegation** — task planning, subagents |
| 5 | **Steering** — human-in-the-loop |
| 6 | **Assignment** — submit a small agent |

Each section below explains **why the feature exists** and **how it works**, then you run a small exercise.

> Self-hosted SDK only. Managed Deep Agents / LangSmith Deployment are out of scope for this lab.


## Part 0 — Setup

**Why this is needed**  
Without a model provider key and a shared `MODEL` string, later cells cannot call OpenRouter. Shared helpers (`show_final`, `show_files`) keep every section readable the same way.

**How it works**  
1. Resolve `LAB_ROOT` from marker files (`AGENTS.md`, `skills/`) and `chdir` there — so Mac/Windows work even if Jupyter’s cwd is wrong.  
2. Install packages from `requirements.txt` with `python -m pip` (same on both OS).  
3. Load `.env` (UTF-8) so `OPENROUTER_API_KEY` is available.  
4. Set `MODEL = "openrouter:openai/gpt-6-sol"`.  

**Cross-platform notes**  
- Use **Python 3.11+** on both Mac and Windows.  
- Agent paths like `/workspace/...` are **virtual POSIX paths** (Deep Agents), not Windows `C:\...` paths — that is intentional with `virtual_mode=True`.  
- Keep the notebook kernel’s files under `deep-agents-lab/` (the setup cell will `chdir` there if it can find the markers).


In [ ]:
# Install deps with the active kernel's interpreter (Mac + Windows).
# Resolves the lab folder via marker files so cwd mistakes don't break -r requirements.txt.
import os
import sys
import subprocess
from pathlib import Path


def find_lab_root() -> Path:
    starts = [Path.cwd().resolve()]
    # VS Code / Cursor Jupyter sometimes expose the notebook path
    for key in ("__vsc_ipynb_file__", "__session__"):
        val = globals().get(key)
        if val:
            starts.append(Path(val).resolve().parent)
    seen = set()
    for start in starts:
        for cand in [start, *start.parents]:
            cand = cand.resolve()
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "AGENTS.md").is_file() and (cand / "skills" / "research-notes" / "SKILL.md").is_file():
                return cand
    raise FileNotFoundError(
        "Could not find deep-agents-lab (expected AGENTS.md and skills/research-notes/SKILL.md). "
        "Open the notebook from inside deep-agents-lab, or set the Jupyter working directory there."
    )


LAB_ROOT = find_lab_root()
os.chdir(LAB_ROOT)
print("LAB_ROOT =", LAB_ROOT)
print("Python  =", sys.version.split()[0], sys.executable)

req = LAB_ROOT / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
print("Dependencies installed.")


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_lab_root() -> Path:
    """Locate deep-agents-lab on Mac/Windows even if cwd is wrong."""
    starts = [Path.cwd().resolve()]
    for key in ("__vsc_ipynb_file__", "__session__"):
        val = globals().get(key)
        if val:
            try:
                starts.append(Path(str(val)).resolve().parent)
            except OSError:
                pass
    seen = set()
    for start in starts:
        for cand in [start, *start.parents]:
            cand = cand.resolve()
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "AGENTS.md").is_file() and (cand / "skills" / "research-notes" / "SKILL.md").is_file():
                return cand
    raise FileNotFoundError(
        "Could not find deep-agents-lab. Open this notebook from that folder "
        "(or set Jupyter's working directory to it)."
    )


LAB_ROOT = find_lab_root()
os.chdir(LAB_ROOT)

# UTF-8 helps Windows consoles print agent text reliably (no-op on modern Mac).
os.environ.setdefault("PYTHONUTF8", "1")
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

# encoding="utf-8" matters if .env was edited in Notepad (avoid locale decode issues)
load_dotenv(LAB_ROOT / ".env", override=True, encoding="utf-8")

assert os.getenv("OPENROUTER_API_KEY"), (
    "OPENROUTER_API_KEY is missing. Copy .env.example to .env and set your key "
    "(Mac: cp .env.example .env | Windows cmd: copy .env.example .env)"
)

MODEL = "openrouter:openai/gpt-6-sol"


def show_final(result):
    """Print the last AI message content."""
    msg = result["messages"][-1]
    content = getattr(msg, "content", msg)
    print(content)
    return content


def show_files(result, limit=20):
    """Print virtual filesystem entries from agent state (if present)."""
    files = result.get("files") or {}
    if not files:
        print("(no files in result state)")
        return
    print(f"{len(files)} file(s) in state:")
    for i, (path, data) in enumerate(files.items()):
        if i >= limit:
            print(f"... and {len(files) - limit} more")
            break
        if isinstance(data, dict) and "content" in data:
            body = data["content"]
            preview = "\n".join(body) if isinstance(body, list) else str(body)
            preview = preview[:200].replace("\n", " / ")
            print(f"  {path}: {preview!r}")
        else:
            print(f"  {path}: {type(data).__name__}")


print("LAB_ROOT =", LAB_ROOT)
print("MODEL    =", MODEL)
print("API key  =", "set" if os.getenv("OPENROUTER_API_KEY") else "MISSING")
print("Platform =", sys.platform)


## Part 1 — What is the Deep Agents harness?

Deep Agents is an **agent harness** on top of LangChain + LangGraph: the same tool-calling loop as other agents, plus built-in capabilities for real multi-step work.

| Group | What you get |
|-------|----------------|
| **Execution environment** | tools, virtual filesystem, permissions, optional code execution, streaming |
| **Context management** | skills, memory, summarization / offloading, prompt caching |
| **Delegation** | opt-in todos (`write_todos`), subagents (`task`) |
| **Steering** | human-in-the-loop interrupts |

Docs: [Overview](https://docs.langchain.com/oss/python/deepagents/overview)

### Quickstart shape

**Why this is needed**  
You need a working baseline before adding middleware, backends, and subagents. If the basic loop fails (auth, model string, tool calling), deeper features will fail too.

**How it works**  
1. You pass `model`, optional `tools`, and `system_prompt` into `create_deep_agent`.  
2. The harness builds a LangGraph graph with a model node + tool node loop.  
3. On `invoke`, the model may call `get_weather`; the harness runs the tool and feeds the result back until the model returns a final answer.  
4. You read `result["messages"][-1]` for the final reply.

Run the cell below (same shape as the overview OpenRouter example).


In [ ]:
# Baseline harness: model + one tool + system_prompt → tool-calling loop
from deepagents import create_deep_agent


def get_weather(city: str) -> str:
    """Get weather for a given city (stub)."""
    return f"It is always sunny in {city}!"


agent = create_deep_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Be concise.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)
show_final(result)


**Takeaway:** `create_deep_agent(model=..., tools=..., system_prompt=...)` is the baseline loop. Later sections only add harness features around this same invoke pattern.


## Part 2 — Execution environment

**Why this group exists**  
An agent that only chats cannot complete real tasks. The execution environment is *where the agent acts*: call APIs, read/write files, optionally run code, and stream progress so you can observe it.

**How the pieces fit**  
The model never touches the outside world directly. It emits tool calls; the harness executes them (custom tools, filesystem tools, interpreter/sandbox) under your configuration (backends, permissions), then returns results into the conversation.


### 2.1 Tools

**Why tools are needed**  
LLMs alone cannot fetch live data or run your business logic. Tools are the controlled bridge to the outside world (APIs, DBs, stubs in this lab).

**How they work**  
1. You define a Python function (or `@tool`) and pass it in `tools=[...]`.  
2. The model sees each tool’s name, description, and argument schema.  
3. When it decides to call a tool, the harness executes your function and appends the return value as a tool message.  
4. The model continues with that evidence.  

MCP is supported the same way (tools from an MCP server) — see [Tools](https://docs.langchain.com/oss/python/deepagents/tools) — but this lab uses local stubs only.


In [ ]:
# Custom tools: model chooses lookup_fact/get_weather from schemas, harness executes them
from langchain.tools import tool


@tool
def lookup_fact(topic: str) -> str:
    """Return a canned fact about a well-known topic."""
    facts = {
        "langgraph": "LangGraph is LangChain's low-level runtime for durable, stateful agent workflows.",
        "deep agents": "Deep Agents is an opinionated harness with planning, filesystem, subagents, and memory.",
        "openrouter": "OpenRouter is a unified API gateway for many LLM providers.",
    }
    key = topic.strip().lower()
    return facts.get(key, f"No canned fact for {topic!r}. Try langgraph, deep agents, or openrouter.")


tools_agent = create_deep_agent(
    model=MODEL,
    tools=[get_weather, lookup_fact],
    system_prompt="Use tools when they help. Prefer lookup_fact for conceptual questions.",
)

result = tools_agent.invoke(
    {"messages": [{"role": "user", "content": "Use lookup_fact for 'deep agents' and summarize in one sentence."}]}
)
show_final(result)


**Takeaway:** Custom tools teach the model *new actions*. Built-in filesystem / `task` tools stay available beside them unless you explicitly restrict them.


### 2.2 Virtual filesystem

**Why a filesystem is needed**  
Long tasks produce notes, drafts, and large intermediates. Stuffing everything into chat blows the context window. A virtual filesystem lets the agent *externalize* working memory with `write_file` / `read_file` / `ls` / `glob` / `grep` (and `delete` / `edit_file`).

**How it works**  
1. Every deep agent gets filesystem tools from `FilesystemMiddleware`.  
2. A **backend** decides where bytes live:  
   - **Default (state)** — files in agent state (`result["files"]`), gone when the run ends unless you checkpoint.  
   - **`FilesystemBackend`** — maps virtual paths onto a real directory (`root_dir` + `virtual_mode=True`).  
3. The model only sees virtual paths like `/notes.md` or `/workspace/hello.md`; you choose the backend.

First cell = in-memory state. Second cell = disk under `./workspace/`.


In [ ]:
# State backend: write_file stores into result['files'] (virtual), not your disk
# Default StateBackend — files live in result["files"], not on disk
fs_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a lab assistant. Use filesystem tools. "
        "Prefer backtick paths like `/notes.md`."
    ),
)

result = fs_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Write /notes.md with the text 'Hello from Deep Agents!', "
                    "then read it back and confirm the contents."
                ),
            }
        ]
    }
)
show_final(result)
print()
show_files(result)


In [ ]:
# FilesystemBackend: same tools, paths map under LAB_ROOT (virtual_mode=True)
from deepagents.backends import FilesystemBackend

workspace = LAB_ROOT / "workspace"
workspace.mkdir(exist_ok=True)

disk_agent = create_deep_agent(
    model=MODEL,
    backend=FilesystemBackend(root_dir=str(LAB_ROOT), virtual_mode=True),
    system_prompt=(
        "You work under a virtual filesystem rooted at the lab folder. "
        "Write practice files under /workspace/ only."
    ),
)

result = disk_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create /workspace/hello.md with one short paragraph about Deep Agents, "
                    "then list /workspace/ with ls."
                ),
            }
        ]
    }
)
show_final(result)
print("\nOn disk:", list(workspace.glob("*.md")))


**Takeaway:** Tool names stay the same; swapping the backend changes *persistence and location*, not the agent’s API for files.


### 2.3 Filesystem permissions

**Why permissions are needed**  
An agent with `write_file` can overwrite secrets or escape a workspace. Declarative permissions give path-level allow/deny *without* rewriting tools.

**How they work**  
1. Pass `permissions=[FilesystemPermission(...), ...]` to `create_deep_agent`.  
2. Each rule has `operations` (`read` / `write`), `paths` (globs), and `mode` (`allow` / `deny` / `interrupt`).  
3. Rules are evaluated **top to bottom; first match wins**. No match → allowed (permissive default).  
4. Only **built-in** filesystem tools are gated — custom tools that open files themselves are not covered.  

Docs: [Permissions](https://docs.langchain.com/oss/python/deepagents/permissions).

In the exercise: deny `/.env` and `/AGENTS.md`, allow `/workspace/**`, deny everything else.


In [ ]:
# Permissions: first-match-wins globs gate built-in fs tools before the tool runs
from deepagents import FilesystemPermission

# Deny secrets first, allow workspace, deny everything else
permissions = [
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/.env", "/.env", "/AGENTS.md"],
        mode="deny",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/**"],
        mode="allow",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/**"],
        mode="deny",
    ),
]

perm_agent = create_deep_agent(
    model=MODEL,
    backend=FilesystemBackend(root_dir=str(LAB_ROOT), virtual_mode=True),
    permissions=permissions,
    system_prompt="Respect filesystem permissions. If a path is denied, report that clearly.",
)

result = perm_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Try to read /.env (or /AGENTS.md). Then write /workspace/safe.txt with 'ok'. "
                    "Report what succeeded and what was denied."
                ),
            }
        ]
    }
)
show_final(result)


**Takeaway:** Put specific denies *before* broad allows. Permissions are static policy; HITL (Part 5) is runtime human judgment for the same class of risk.


### 2.4 Code execution (lightweight)

**Why code execution is needed**  
Some steps are better as deterministic code (loops, transforms, batching) than as another model turn. Sandboxes also unlock shell/CLI workflows.

**How the two paths differ**

| Need | Mechanism | How it runs |
|------|-----------|-------------|
| Shell, installs, OS FS | [Sandbox backend](https://docs.langchain.com/oss/python/deepagents/sandboxes) | Backend exposes `execute`; commands run in an isolated environment |
| In-process JS loops / transforms | [Interpreter](https://docs.langchain.com/oss/python/deepagents/interpreters) (`CodeInterpreterMiddleware`) | Agent gets an `eval`-style tool; QuickJS runs in-process; intermediates stay out of the model context |

**This lab:** try the QuickJS interpreter if installed (`deepagents[quickjs]`). Sandboxes need extra infra — skipped here on purpose.


In [ ]:
# Interpreter middleware: optional in-process JS eval tool (no Docker sandbox)
try:
    from langchain_quickjs import CodeInterpreterMiddleware

    interp_agent = create_deep_agent(
        model=MODEL,
        middleware=[CodeInterpreterMiddleware()],
        system_prompt=(
            "You can use the code interpreter (eval) for small JavaScript calculations. "
            "Prefer eval for arithmetic."
        ),
    )
    result = interp_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Use the interpreter to compute the sum of 1..20 and report the number.",
                }
            ]
        }
    )
    show_final(result)
except ImportError as e:
    print("Interpreter skipped — install with: pip install 'deepagents[quickjs]'")
    print("Error:", e)
    print("Sandbox execute is also out of scope for this lab (see Sandboxes docs).")


**Takeaway:** Interpreters compose tools/data *inside* the agent loop; sandboxes act on an *environment*. Pick based on whether you need OS access.


### 2.5 Streaming

**Why streaming is needed**  
`invoke` only returns at the end. For UIs, debugging, and HITL, you need progressive tokens, tool calls, and state updates.

**How it works**  
1. Call `agent.stream(..., stream_mode=["messages", "updates"])` (or other modes).  
2. **`messages`** yields token/message chunks as the model writes.  
3. **`updates`** yields graph/node state patches (including tool activity).  
4. Deep Agents can also expose subagent streams (`stream.subagents`) when work is delegated.  

Docs: [Event streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming).


In [ ]:
# stream(): observe tokens (messages) and graph patches (updates) live
stream_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_fact],
    system_prompt="Use lookup_fact when helpful. Be brief.",
)

print("=== stream (messages) ===")
for mode, data in stream_agent.stream(
    {"messages": [{"role": "user", "content": "lookup_fact for langgraph, one sentence."}]},
    stream_mode=["messages", "updates"],
):
    if mode == "messages":
        token, _meta = data
        text = getattr(token, "content", None) or ""
        if text:
            print(text, end="", flush=True)
    elif mode == "updates" and isinstance(data, dict):
        keys = [k for k in data.keys() if k != "__interrupt__"]
        if keys:
            print(f"\n[update keys: {keys}]")
print()


**Takeaway:** Streaming does not change what the agent *can* do — it changes how *you observe* the execution environment in real time.


## Part 3 — Context management

**Why this group exists**  
Token limits and noisy history kill long-running agents. Context management controls *what the agent knows at startup*, *what it loads on demand*, and *how it compresses* as the run grows.

| Layer | Role |
|-------|------|
| Skills | On-demand specialized instructions (`SKILL.md`) |
| Memory | Always-on preferences (`AGENTS.md`) |
| Summarization / offloading | Compress history; park large tool results in files |
| Prompt caching | Reuse static prompt prefixes on supported providers |


### 3.1 Skills

**Why skills are needed**  
Putting every playbook in `system_prompt` wastes tokens and confuses the model. Skills package domain workflows and load **only when relevant** (progressive disclosure).

**How they work**  
1. Each skill is a folder with `SKILL.md` (YAML frontmatter: `name`, `description`).  
2. Pass `skills=["./skills/"]` and a filesystem-capable `backend` (e.g. `FilesystemBackend`).  
3. At startup the agent sees skill *metadata*; when a task matches, it reads the full skill body from the backend.  
4. Supporting files (templates, scripts) can live beside `SKILL.md`.  

This lab’s skill: `skills/research-notes/`.


In [ ]:
# Skills: progressive disclosure from ./skills via filesystem backend
skills_agent = create_deep_agent(
    model=MODEL,
    backend=FilesystemBackend(root_dir=str(LAB_ROOT), virtual_mode=True),
    skills=["./skills/"],
    permissions=[
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/workspace/**", "/skills/**"],
            mode="allow",
        ),
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/**"],
            mode="deny",
        ),
    ],
    system_prompt=(
        "You are a research assistant. When asked to research or take notes, "
        "use the research-notes skill if available."
    ),
)

result = skills_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Using the research-notes skill, draft brief notes about what a Deep Agent harness is. "
                    "Write the notes file under /workspace/ and give me the executive summary."
                ),
            }
        ]
    }
)
show_final(result)


**Takeaway:** Skills = *optional* deep instructions. Memory (next) = *always-on* preferences. Don’t mix the two roles.


### 3.2 Memory (`AGENTS.md`)

**Why memory is needed**  
Users expect stable preferences (tone, paths, formatting) across turns without restating them every message.

**How it works**  
1. Pass `memory=["/AGENTS.md"]` or `memory=["./AGENTS.md"]` depending on backend.  
2. Unlike skills, memory content is **loaded every run** into the agent’s context.  
3. With the default state backend, seed files via invoke `files={...: create_file_data(...)}`.  
4. Add a **checkpointer** + stable `thread_id` so conversation *and* durable state continue across `invoke` calls.  

The agent may also update memory files when you allow writes — treat that as persistent policy evolution.


In [ ]:
# Memory: AGENTS.md always loaded; checkpointer+thread_id keep the session
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import MemorySaver

agents_md = (LAB_ROOT / "AGENTS.md").read_text(encoding="utf-8")
memory_checkpointer = MemorySaver()

memory_agent = create_deep_agent(
    model=MODEL,
    memory=["/AGENTS.md"],
    checkpointer=memory_checkpointer,
    system_prompt="Follow preferences in your memory files.",
)

config = {"configurable": {"thread_id": "memory-lab-1"}}

result = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What preferences are in your memory? Answer in the style those prefs request.",
            }
        ],
        "files": {"/AGENTS.md": create_file_data(agents_md)},
    },
    config=config,
)
show_final(result)

# Same thread — memory + conversation continuity
result2 = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Using those preferences, give me 3 bullets on why subagents help.",
            }
        ]
    },
    config=config,
)
print("\n--- follow-up ---")
show_final(result2)


**Takeaway:** Memory shapes *every* turn; checkpointer + `thread_id` make multi-turn sessions real instead of isolated invokes.


### 3.3 Summarization and context offloading

**Why this is needed**  
Tool outputs and chat history grow without bound. Without compression, the model hits context limits or becomes expensive/noisy mid-task.

**How it works** (harness-level)  
1. **Input context** — system prompt, memory, skills, tool prompts.  
2. **Offloading** — large tool results can be written to the virtual filesystem and referenced by path.  
3. **Summarization** — older conversation can be compressed so only what matters stays in the window.  
4. **Isolation** — subagents (Part 4) return a short report instead of their full trace.  

Docs: [Context engineering](https://docs.langchain.com/oss/python/deepagents/context-engineering).

Exercise: a deliberately large `fetch_large_report` tool; the agent should summarize to a file and answer briefly.


In [ ]:
# Offloading demo: large tool payload → agent should summarize via filesystem
@tool
def fetch_large_report(topic: str) -> str:
    """Return a large synthetic report for context-offloading practice."""
    line = f"Detail about {topic}: Deep Agents manage context with files, summarization, and subagents. "
    # Large-ish payload to encourage offloading / careful handling
    return (line * 400) + f"\n\nEND REPORT on {topic}"


offload_agent = create_deep_agent(
    model=MODEL,
    tools=[fetch_large_report],
    system_prompt=(
        "When tool results are large, write a condensed summary to /summary.md using filesystem tools "
        "and answer from that summary. Do not paste the full report in the final reply."
    ),
)

result = offload_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Call fetch_large_report for 'context offloading' and give me a 3-bullet summary only.",
            }
        ]
    }
)
show_final(result)
print()
show_files(result)


**Takeaway:** Prefer files + short answers over pasting megabytes into chat. The filesystem is working memory for long runs.


### 3.4 Prompt caching (note only)

**Why prompt caching exists**  
Static prefixes (base instructions, memory, skills) repeat every turn. Caching them cuts latency and cost on long agents.

**How it works**  
On **Anthropic** and **Amazon Bedrock**, `create_deep_agent` automatically marks those static sections cache-eligible. No extra config for those providers.

**This lab:** OpenRouter + GPT-6 Sol — Anthropic/Bedrock automatic caching does **not** apply. See the [overview](https://docs.langchain.com/oss/python/deepagents/overview#prompt-caching) if you switch models.


## Part 4 — Delegation

**Why delegation exists**  
One context window cannot cleanly hold a huge plan *and* all intermediate research. Delegation splits work into trackable steps (todos) and isolated workers (subagents).

| Layer | Job |
|-------|-----|
| Task planning | Structured todo list in agent state |
| Subagents | Ephemeral child agents via the `task` tool |


### 4.1 Task planning

**Why planning is needed**  
Multi-step requests fail silently when the model skips steps. An explicit todo list makes progress visible (and streamable in UIs).

**How it works**  
1. Since deepagents **v0.7**, planning is **opt-in**: pass `middleware=[TodoListMiddleware()]`.  
2. That exposes `write_todos` with statuses like `pending` / `in_progress` / `completed`.  
3. Todos live in agent **state** (inspect `result.get("todos")`).  
4. Useful for long tasks, weaker models, and progress UIs.  

Without the middleware, the agent can still “plan” in prose — but you lose structured state.


In [ ]:
# TodoListMiddleware opts into write_todos + structured todos in state
from langchain.agents.middleware import TodoListMiddleware

plan_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_fact, get_weather],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "For multi-step requests, use write_todos to plan, then execute the steps. "
        "Mark todos completed as you go."
    ),
)

result = plan_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Do these steps: (1) lookup_fact for langgraph, "
                    "(2) get_weather for Tokyo, "
                    "(3) write a 2-sentence combined answer. Use todos."
                ),
            }
        ]
    }
)
show_final(result)
todos = result.get("todos")
print("\nTodos in state:", todos)


**Takeaway:** Todos are harness state, not just chat text — add the middleware when you need accountability or UI progress.


### 4.2 Subagents

**Why subagents are needed**  
Heavy research pollutes the supervisor’s context. A subagent runs in isolation (by default), then returns **one** report the parent can synthesize.

**How they work**  
1. The harness provides a built-in `task` tool.  
2. You register custom specs: `name`, `description`, `system_prompt`, optional `tools` / `model` / `permissions`.  
3. Default **`mode: "isolated"`** — child sees only the delegated task, not the full parent history. (`fork` can inherit history when needed.)  
4. Sync subagents block until done; async subagents (not exercised here) support parallel launch/check/cancel.  

Docs: [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents).


In [ ]:
# Subagent via task tool: isolated child runs mock_search, returns one report
@tool
def mock_search(query: str) -> str:
    """Mock web search for lab use (no external API)."""
    q = query.lower()
    if "langgraph" in q:
        return (
            "LangGraph is a low-level orchestration framework for stateful, multi-actor "
            "applications with LLMs. It powers durable execution, branching, and HITL."
        )
    if "deep agent" in q:
        return (
            "Deep Agents add planning, filesystem context, subagents, and memory on top "
            "of LangChain/LangGraph."
        )
    return f"Mock results for {query!r}: [result A], [result B]"


research_subagent = {
    "name": "research-agent",
    "description": "Delegate in-depth research questions that need mock_search.",
    "system_prompt": (
        "You are a focused researcher. Use mock_search, then return a short factual report "
        "with 3-5 bullets. Do not call other agents."
    ),
    "tools": [mock_search],
}

supervisor = create_deep_agent(
    model=MODEL,
    subagents=[research_subagent],
    system_prompt=(
        "You are a supervisor. For research questions, delegate to research-agent via the task tool, "
        "then synthesize a brief final answer for the user."
    ),
)

result = supervisor.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Research what LangGraph is and give me a short briefing.",
            }
        ]
    }
)
show_final(result)


**Takeaway:** Use subagents for isolation and token efficiency; keep the supervisor focused on routing and final synthesis.


## Part 5 — Steering (human-in-the-loop)

**Why HITL is needed**  
Permissions block whole path classes. Humans still need to approve *specific* risky calls (writes, deletes, expensive APIs) at runtime.

**How it works**  
1. Pass `interrupt_on={ "write_file": {...} }` (tool name → bool or config).  
2. **Requires a checkpointer** so the graph can pause and later resume.  
3. Use a stable `thread_id` in `config`.  
4. When a matching tool is about to run, the invoke returns `__interrupt__` with `action_requests`.  
5. Resume with `Command(resume={"decisions": [{"type": "approve"|"edit"|"reject"}, ...]})` on the **same** config.  

Docs: [Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop).

Permissions (Part 2.3) = static rules. HITL = interactive steering for the same risk class.


In [ ]:
# HITL: interrupt_on pauses write_file; Command(resume=...) continues same thread
from langchain_core.utils.uuid import uuid7
from langgraph.types import Command

hitl_checkpointer = MemorySaver()

hitl_agent = create_deep_agent(
    model=MODEL,
    interrupt_on={
        "write_file": {
            "allowed_decisions": ["approve", "edit", "reject"],
        }
    },
    checkpointer=hitl_checkpointer,
    system_prompt="When asked to write a file, use write_file. Be concise.",
)

config = {"configurable": {"thread_id": str(uuid7())}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Write /approved.txt containing 'HITL lab ok'.",
            }
        ]
    },
    config=config,
)

if result.get("__interrupt__"):
    interrupt_value = result["__interrupt__"][0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = interrupt_value["review_configs"]
    config_map = {cfg["action_name"]: cfg for cfg in review_configs}

    print("Paused for human review:")
    for action in action_requests:
        review_config = config_map[action["name"]]
        print(f"  tool={action['name']} args={action['args']}")
        print(f"  allowed={review_config['allowed_decisions']}")

    # Lab default: approve all pending writes
    decisions = [{"type": "approve"} for _ in action_requests]
    result = hitl_agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
    )
else:
    print("No interrupt raised (model may have answered without write_file).")

print("\nFinal:")
show_final(result)
show_files(result)


**Takeaway:** Checkpointer + `thread_id` + `Command(resume=...)` turn a dangerous tool call into a reviewable workflow instead of a blind side effect.


## Part 6 — Assignment (submit this)

Use what you practiced in Parts 1–5 to build **one** small deep agent and run it.

### Requirements (keep it small)

1. Create a deep agent with **at least 3** of these:
   - custom tool (`mock_search` is fine)
   - `TodoListMiddleware`
   - `FilesystemBackend` writing under `/workspace/`
   - `permissions`
   - `skills` and/or `memory`
2. Ask the agent: **What is LangGraph?**
3. The agent must write a short report to **`/workspace/submission.md`**.
4. Show the final answer with `show_final(result)`.

### What to submit

- This notebook with Part 6 cell **output visible**
- The generated file `workspace/submission.md`

Fill in the TODO cell below (starter code is incomplete on purpose).


In [ ]:
# === ASSIGNMENT: complete the TODOs, then run ===

# You may reuse helpers from earlier cells: MODEL, LAB_ROOT, mock_search,
# FilesystemBackend, FilesystemPermission, TodoListMiddleware, show_final

# TODO 1) system prompt — tell the agent to research and write /workspace/submission.md
system_prompt = """TODO: write a short research system prompt"""

# TODO 2) build the agent (include at least 3 features from the requirements list)
agent = create_deep_agent(
    model=MODEL,
    # tools=...,
    # system_prompt=system_prompt,
    # middleware=...,
    # backend=...,
    # memory=...,
    # skills=...,
    # permissions=...,
)

# TODO 3) invoke with: What is LangGraph?
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "TODO: ask the required question and require /workspace/submission.md",
            }
        ]
    }
)

show_final(result)

submission = LAB_ROOT / "workspace" / "submission.md"
print("submission.md exists:", submission.exists())
if submission.exists():
    print(submission.read_text(encoding="utf-8")[:500])


### Submission checklist

- [ ] Part 6 cell runs without errors  
- [ ] `workspace/submission.md` exists and mentions LangGraph  
- [ ] Your `create_deep_agent(...)` uses at least 3 lab features  

Optional reading: [Customization](https://docs.langchain.com/oss/python/deepagents/customization) · [Overview](https://docs.langchain.com/oss/python/deepagents/overview)
